In [1]:
from typing import List, TypedDict
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

d:\Coding\Genarative-AI\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=175)

chunks = text_splitter.split_documents(docs)


for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [6]:
print(len(chunks))

7456


In [7]:
embedding = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6250.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
vector_store = FAISS.from_documents(chunks,embedding)

In [10]:
llm = HuggingFaceEndpoint(
    repo_id = 'zai-org/GLM-4.6',
    task = 'text-generation'
)

In [ ]:
class State(TypedDict):
    question = str
    docs = list[Document]
    

    strips = list[str]
    kept_strips: list[str]
    refined_context: str

    answer = str

In [9]:
def retrieve(state):
    q = state['question']
    return {'docs': retriever.invoke(q)}

In [ ]:
# retriever sentence striper funtion
def decompose_to_sentence(text: str) -> list[str]:
    text = re.sub(r"\s+"," ",text).strip()
    sentence = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentence if len(s.strip()) > 20]

# Filter
class KeepOrDrop(BaseModel):
    keep:bool

filter_prompt = ChatpromptTemplate.from_messages(
    [
        ("system","you are a strip relevance fiter . \n"
        "Return keep=true only if the sentence directly helps answer the queston. \n"
        "Use ONLY the sentence. Output JSON only"
        ),
        ("human","Question: {question}\n\nSentence:\n{sentence}")


    ]
)

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)


# refining (decompose -> filter -> recompose)
def refine(state: State) -> State:
    q = state["question"]
    # Combine retrieve docs into the one contex string
    context = "\n\n".join(d.page_content for d in state["docs"]).strip()

    #1 DECOMPOSITION: contex -> sentence strips
    strips = decompose_to_sentence(context)

    #2 FILTER: keep only relevance strips
    kept: list[str] = []
    
    for s in strips:
        if filter_chain.invoke({'question':q,"sentence":s}).keep:
            kept.append(s)

    #3 RECOMPOSE: strips merge
    refined_context = "\n".join(kept).strip()

    return{
        "strips":strips,
        "kept_strips": kept,
        "refined_context":refined_context
    }


In [ ]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',"Answer only from the context. If not in contex, say you don't know",),
        ('human', "Question : {question}\n\nContext:\n{context}")
    ]
)

def generate(state:State) -> State:
    
    out = (answer_prompt | llm).invoke({"question": state["question"], "refine_context": state['refined_context']})
    return {"answer": out.content}

In [ ]:
g = StateGraph(State)
g.add_node("retrieve",retrieve)
g.add_node("refine", refine)
g.add_node("generate", generate)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
app = g.compile()

In [ ]:
res = app.invoke({"question":"whte is a transformer","docs":[],'answer':""})
print(res["answer"])

In [ ]:
print(res["docs"][0].page_content)
print('*'*100)
print(res["docs"][1].page_content)
print('*'*100)
print(res["docs"][2].page_content) 
print('*'*100)
print(res["docs"][3].page_content)
print('*'*100)